# Laboratorio 7 — Spark MLlib
---

## 0. Configuración inicial

Importamos librerías, iniciamos la sesión de Spark y definimos rutas y parámetros globales del notebook.

In [1]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.stat import Correlation
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark.ml import Pipeline

spark = (SparkSession.builder
         .appName("Lab7_SparkMLlib_ENEIC")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark


In [5]:
# ------------------------------------------------------------------
# RUTAS Y PARÁMETROS GLOBALES
# ------------------------------------------------------------------
RAW_DIR = "../data/raw"          # carpeta con los .xlsx originales de la ENEIC
PARQUET_DIR = "../data/parquet"  # carpeta de salida para los datos preparados
SAMPLE_MAX = 8000             # tamaño máximo de muestra SOLO para graficar
SEED = 42

os.makedirs(PARQUET_DIR, exist_ok=True)

FILE_MAP = {
    "2025T1": {"path": f"{RAW_DIR}/Personas_2025_T1.xlsx", "anio": 2025, "trim_cal": 1, "uso": "train"},
    "2025T2": {"path": f"{RAW_DIR}/Personas_2025_T2.xlsx", "anio": 2025, "trim_cal": 2, "uso": "train"},
    "2025T3": {"path": f"{RAW_DIR}/Personas_2025_T3.xlsx", "anio": 2025, "trim_cal": 3, "uso": "train"},
    "2025T4": {"path": f"{RAW_DIR}/Personas_2025_T4.xlsx", "anio": 2025, "trim_cal": 4, "uso": "train"},
    "2026T1": {"path": f"{RAW_DIR}/Personas_2026_T1.xlsx", "anio": 2026, "trim_cal": 1, "uso": "test"},
}

# Columnas originales que se conservan del archivo fuente (ver tabla "Variables que utilizará")
REQUIRED_COLS = [
    "NUM_HOGAR", "NUM_PERSONA",      # auditoría de registros
    "FACTOR",                         # diseño muestral (no se usa para ponderar en este lab)
    "ANIO", "TRIMESTRE",              # auditoría de la fuente (TRIMESTRE NO es el calendario)
    "OCUPADOS",                       # filtro
    "P05C16",                         # categoria_ocupacional (filtro + predictor)
    "P05D01",                         # salario_mensual (variable objetivo)
    "P02A03",                         # edad
    "P05C07A", "P05C07B",             # antiguedad_anios, antiguedad_meses
    "P05H01A",                        # horas_semanales
    "P03A03A",                        # nivel_educativo
    "DOMINIO",                        # dominio
]


## 1. Carga, armonización y calidad de datos — (5 pts)

**Enfoque:**
1. Cada archivo Excel se lee con `pandas`/`openpyxl` **como texto (`dtype=str`)**. Esto es
   clave: en la ENEIC un mismo código puede llegar como número (`1`) o como texto con
   decimales espurios (`"1.0"`), y leer todo como texto nos permite normalizarlo de forma
   uniforme *antes* de convertir a Spark, en vez de arrastrar inconsistencias de tipo.
2. Se seleccionan únicamente las columnas requeridas (esto resuelve el problema de que el
   archivo de IV-2025 tiene 302 columnas en vez de 270: al elegir el mismo subconjunto de
   columnas en todos los archivos, `unionByName` los puede unir sin importar cuántas
   columnas tenía cada archivo originalmente).
3. Se agregan `archivo_origen`, `periodo_archivo`, `anio_archivo` y `trimestre_calendario`
   a partir del archivo de procedencia (NO a partir de la columna `TRIMESTRE`, que trae
   valores inconsistentes entre archivos: 2, 3, 4, 5 y 6, y que en II-2025 mezcla 3 y 2).
4. Se homologan los códigos categóricos a una representación de texto consistente
   (quitando sufijos `".0"` que aparecen cuando el valor llegó como numérico).
5. Se hace `unionByName` de los cuatro trimestres de 2025.


In [ ]:
def leer_archivo_personas(periodo, path, anio, trim_cal):
    """Lee un archivo de Personas de la ENEIC, selecciona columnas requeridas y
    agrega las columnas de identificación de período. Devuelve (spark_df, n_original)."""
    pdf = pd.read_excel(path, dtype=str)
    faltan = [c for c in REQUIRED_COLS if c not in pdf.columns]
    if faltan:
        raise ValueError(f"[{periodo}] Faltan columnas requeridas: {faltan}")

    n_original = len(pdf)
    pdf = pdf[REQUIRED_COLS].copy()

    sdf = spark.createDataFrame(pdf)
    sdf = (sdf
           .withColumn("archivo_origen", F.lit(periodo))
           .withColumn("periodo_archivo", F.lit(periodo))
           .withColumn("anio_archivo", F.lit(int(anio)))
           .withColumn("trimestre_calendario", F.lit(int(trim_cal))))
    return sdf, n_original


def normalizar_codigo(colname):
    """Convierte una columna de texto a una representación de código consistente:
    quita espacios y el sufijo '.0' que aparece cuando el dato llegó como numérico."""
    c = F.trim(F.col(colname).cast("string"))
    c = F.regexp_replace(c, r"\.0$", "")
    return F.when((c == "") | c.isNull(), None).otherwise(c)


# --- Cargar los 4 trimestres de 2025 ---
dfs_2025 = []
registros_originales = {}
for periodo in ["2025T1", "2025T2", "2025T3", "2025T4"]:
    info = FILE_MAP[periodo]
    sdf, n_orig = leer_archivo_personas(periodo, info["path"], info["anio"], info["trim_cal"])
    dfs_2025.append(sdf)
    registros_originales[periodo] = n_orig

df_2025 = dfs_2025[0]
for d in dfs_2025[1:]:
    df_2025 = df_2025.unionByName(d)

df_2025.persist()
print("Registros originales por archivo (2025):", registros_originales)
print("Total tras unionByName:", df_2025.count())


In [12]:
import os, sys
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

spark = (SparkSession.builder
         .appName("Lab7_SparkMLlib_ENEIC")
         .master("local[*]")
         .config("spark.driver.host", "127.0.0.1")
         .config("spark.driver.bindAddress", "127.0.0.1")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())

In [ ]:
# --- Esquema y 5 registros de las columnas seleccionadas ---
df_2025.printSchema()
df_2025.select(*REQUIRED_COLS, "archivo_origen", "periodo_archivo",
               "anio_archivo", "trimestre_calendario").show(5, truncate=False)


In [ ]:
# --- Cargar también I-2026 (se preparará igual, pero se guarda por separado: es TEST) ---
info_2026 = FILE_MAP["2026T1"]
df_2026, n_orig_2026 = leer_archivo_personas("2026T1", info_2026["path"],
                                              info_2026["anio"], info_2026["trim_cal"])
df_2026.persist()
registros_originales["2026T1"] = n_orig_2026
print("Registros originales I-2026:", n_orig_2026)


### Homologación de tipos y armonización de códigos

Se castean las variables numéricas a `double` y se normalizan las variables categóricas
(`P05C16`, `P03A03A`, `DOMINIO`, `OCUPADOS`, `NUM_HOGAR`, `NUM_PERSONA`) a texto consistente
con `normalizar_codigo`. El código educativo `"0"` (ninguno) se conserva explícitamente y
**no** se trata como faltante.


In [ ]:
def armonizar_tipos(sdf):
    sdf = (sdf
        .withColumn("NUM_HOGAR", normalizar_codigo("NUM_HOGAR"))
        .withColumn("NUM_PERSONA", normalizar_codigo("NUM_PERSONA"))
        .withColumn("OCUPADOS", normalizar_codigo("OCUPADOS"))
        .withColumn("categoria_ocupacional", normalizar_codigo("P05C16"))
        .withColumn("nivel_educativo", normalizar_codigo("P03A03A"))
        .withColumn("dominio", normalizar_codigo("DOMINIO"))
        .withColumn("salario_mensual", F.col("P05D01").cast(T.DoubleType()))
        .withColumn("edad", F.col("P02A03").cast(T.DoubleType()))
        .withColumn("antiguedad_anios", F.col("P05C07A").cast(T.DoubleType()))
        .withColumn("antiguedad_meses", F.col("P05C07B").cast(T.DoubleType()))
        .withColumn("horas_semanales", F.col("P05H01A").cast(T.DoubleType()))
        .withColumn("FACTOR", F.col("FACTOR").cast(T.DoubleType()))
        .withColumn("ANIO", F.col("ANIO").cast(T.IntegerType()))
        .withColumn("TRIMESTRE", F.col("TRIMESTRE").cast(T.IntegerType()))
    )
    return sdf

df_2025_tip = armonizar_tipos(df_2025)
df_2026_tip = armonizar_tipos(df_2026)
df_2025_tip.persist()
df_2026_tip.persist()
df_2025_tip.select("categoria_ocupacional", "nivel_educativo", "dominio",
                    "salario_mensual", "edad", "antiguedad_anios",
                    "antiguedad_meses", "horas_semanales").show(5)


### Faltantes por variable (antes de aplicar filtros)

Se calcula cantidad y porcentaje de nulos/vacíos para cada variable clave, **sobre la unión
de 2025 sin filtrar todavía**.


In [ ]:
KEY_COLS = ["salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses",
            "horas_semanales", "nivel_educativo", "categoria_ocupacional",
            "dominio", "OCUPADOS"]

n_total_2025 = df_2025_tip.count()
filas_faltantes = []
for c in KEY_COLS:
    n_null = df_2025_tip.filter(F.col(c).isNull() | (F.col(c).cast("string") == "")).count()
    filas_faltantes.append((c, n_null, round(100 * n_null / n_total_2025, 2)))

faltantes_pdf = pd.DataFrame(filas_faltantes, columns=["variable", "n_faltantes", "pct_faltantes"])
faltantes_pdf


### Filtros de población analítica

Se aplican en un **orden fijo** para poder contabilizar cuántos registros se excluyen en
cada paso. Nótese el uso explícito de `isnan()` (además de `isNotNull()`): en Spark, un
valor `NaN` se considera *mayor* que cualquier otro número, por lo que un filtro como
`salario_mensual > 0` **no** excluye los `NaN` por sí solo. Por eso se define un helper
`es_finito` que exige que el valor no sea nulo, no sea `NaN` y no sea `+/-Infinito`.


In [ ]:
def es_finito(colname):
    c = F.col(colname)
    return c.isNotNull() & (~F.isnan(c)) & (c != float("inf")) & (c != float("-inf"))


def preparar_poblacion_analitica(sdf, etiqueta):
    """Aplica los filtros de la población analítica en orden fijo, registrando
    cuántos registros se excluyen en cada paso."""
    pasos = [("0_inicial", sdf.count())]
    d = sdf

    d = d.filter(es_finito("edad") & (F.col("edad") >= 15))
    pasos.append(("1_edad>=15_finita", d.count()))

    d = d.filter(F.col("OCUPADOS") == "1")
    pasos.append(("2_ocupado==1", d.count()))

    d = d.filter(F.col("categoria_ocupacional").isin(["1", "2", "3", "4"]))
    pasos.append(("3_asalariado_P05C16_1a4", d.count()))

    d = d.filter(es_finito("salario_mensual") & (F.col("salario_mensual") > 0))
    pasos.append(("4_salario_finito_positivo", d.count()))

    d = d.filter(es_finito("antiguedad_meses")
                 & (F.col("antiguedad_meses") == F.floor(F.col("antiguedad_meses")))
                 & (F.col("antiguedad_meses") >= 0) & (F.col("antiguedad_meses") <= 11))
    pasos.append(("5_meses_0_a_11_enteros", d.count()))

    d = d.filter(es_finito("antiguedad_anios") & (F.col("antiguedad_anios") >= 0))
    pasos.append(("6_antiguedad_anios>=0", d.count()))

    d = d.withColumn("antiguedad", F.col("antiguedad_anios") + F.col("antiguedad_meses") / F.lit(12.0))
    d = d.filter(F.col("antiguedad") <= F.col("edad"))
    pasos.append(("7_antiguedad<=edad", d.count()))

    d = d.filter(es_finito("horas_semanales")
                 & (F.col("horas_semanales") > 0) & (F.col("horas_semanales") <= 168))
    pasos.append(("8_horas_0_a_168", d.count()))

    # Categóricas ausentes o no reconocidas -> "DESCONOCIDO" (0 en nivel_educativo NO es faltante)
    # Fuente: Diccionario Personas ENEIC (I-2025, II-2025, III-2025, IV-2025, I-2026) — códigos idénticos en los 5 archivos
    codigos_educativos_validos = ["0", "1", "2", "3", "4", "5", "6", "7"]
    # 0=NINGUNO, 1=PREPRIMARIA, 2=PRIMARIA, 3=BÁSICO, 4=DIVERSIFICADO, 5=SUPERIOR, 6=MAESTRÍA, 7=DOCTORADO
    dominios_validos = ["1", "2", "3"]
    # 1=Urbano Metropolitano, 2=Resto Urbano, 3=Rural Nacional

    d = d.withColumn("nivel_educativo",
                      F.when(F.col("nivel_educativo").isin(codigos_educativos_validos),
                             F.col("nivel_educativo"))
                       .otherwise(F.lit("DESCONOCIDO")))
    d = d.withColumn("dominio",
                      F.when(F.col("dominio").isin(dominios_validos), F.col("dominio"))
                       .otherwise(F.lit("DESCONOCIDO")))

    print(f"--- Pasos de filtrado ({etiqueta}) ---")
    for nombre, n in pasos:
        print(f"  {nombre}: {n}")

    return d, pasos


df_2025_analitico, pasos_2025 = preparar_poblacion_analitica(df_2025_tip, "2025 - train/dev")
df_2026_analitico, pasos_2026 = preparar_poblacion_analitica(df_2026_tip, "2026 - test final")

df_2025_analitico.persist()
df_2026_analitico.persist()

pasos_2025_pdf = pd.DataFrame(pasos_2025, columns=["paso", "n_registros"])
pasos_2025_pdf["n_excluidos"] = pasos_2025_pdf["n_registros"].shift(1) - pasos_2025_pdf["n_registros"]
pasos_2025_pdf


**Nota:** los códigos válidos usados arriba para `nivel_educativo` y `dominio` son un
placeholder (`0`–`9` y `1`–`8` respectivamente). **Deben reemplazarse por los códigos reales
del diccionario de datos de la ENEIC** (P03A03A y DOMINIO) antes de la entrega final, para
que la marca `DESCONOCIDO` sea precisa y no capture códigos válidos por error.

### Verificación de unicidad de `(periodo_archivo, NUM_HOGAR, NUM_PERSONA)`

In [ ]:
claves = df_2025_analitico.groupBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA").count()
duplicados = claves.filter(F.col("count") > 1)
n_dup = duplicados.count()
print(f"Combinaciones de clave con más de 1 registro: {n_dup}")

if n_dup > 0:
    # Inspeccionar si son repeticiones EXACTAS (todas las columnas iguales) o registros en conflicto
    llaves_dup = duplicados.select("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA")
    filas_dup = df_2025_analitico.join(llaves_dup, on=["periodo_archivo", "NUM_HOGAR", "NUM_PERSONA"])
    n_filas_dup = filas_dup.count()
    n_filas_dup_exactas = filas_dup.distinct().count()
    print(f"Filas involucradas: {n_filas_dup} | Filas distintas tras comparar TODAS las columnas: {n_filas_dup_exactas}")
    if n_filas_dup_exactas < n_filas_dup:
        print("=> Hay repeticiones EXACTAS de fila completa (posible doble carga).")
    if n_filas_dup_exactas == n_filas_dup:
        print("=> Son registros EN CONFLICTO (misma clave, columnas distintas): requiere investigación manual.")
    filas_dup.orderBy("periodo_archivo", "NUM_HOGAR", "NUM_PERSONA").show(20, truncate=False)
else:
    print("No se encontraron combinaciones duplicadas de (periodo_archivo, NUM_HOGAR, NUM_PERSONA).")


### Guardar en Parquet

In [ ]:
df_2025_analitico.write.mode("overwrite").parquet(f"{PARQUET_DIR}/personas_2025_analitico.parquet")
df_2026_analitico.write.mode("overwrite").parquet(f"{PARQUET_DIR}/personas_2026_analitico.parquet")
print("Guardado en:", PARQUET_DIR)


### Respuestas — Ejercicio 1

**¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?**
Porque tiene 302 columnas originales frente a las 270 de los demás trimestres: el orden y
significado de las columnas por posición no coincide entre archivos. Apilar por posición
(`union`) mezclaría columnas distintas bajo el mismo nombre de columna. Por eso se usa
`unionByName` sobre un subconjunto fijo de columnas seleccionadas por **nombre**, no por
posición.

**¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una
respuesta no registrada?**
Un dato ausente "por no corresponder" ocurre cuando la pregunta no aplica a esa persona
dado el flujo del cuestionario (por ejemplo, preguntas de antigüedad laboral no se le hacen
a alguien que no trabaja); es un vacío *estructural* y esperado. Una respuesta no registrada
es un vacío en una pregunta que sí correspondía a la persona, pero que no quedó capturada
(el encuestado no respondió, error de captura, etc.); es un faltante *real* que sí afecta la
calidad del dato para ese registro. Tratarlos igual (por ejemplo, imputando ambos a 0)
distorsiona el análisis.

**¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del
conjunto longitudinal?**
Porque la ENEIC tiene diseño de panel rotativo: la misma persona puede ser entrevistada en
trimestres consecutivos como parte del diseño muestral. Esas observaciones repetidas de la
misma persona en distintos períodos son observaciones legítimas (representan su situación en
cada corte), no errores de captura. Un verdadero duplicado sería la misma persona repetida
dentro de un mismo `periodo_archivo`.

**¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores
del país?**
Porque (a) es una muestra encuestal, no un censo, y solo mediante el factor de expansión
(`FACTOR`) se puede proyectar a la población total; en este laboratorio se trabaja de forma
**no ponderada**, así que los conteos describen únicamente a las personas encuestadas que
cumplen los filtros, y (b) los filtros aplicados (edad, ocupación, asalariados, salario
positivo, consistencia de antigüedad/horas) excluyen deliberadamente a otros grupos de la
población económicamente activa (independientes, no ocupados, registros inconsistentes).


## 2. Estadística descriptiva y preguntas de exploración — (5 pts)

Todas las métricas y estadísticas se calculan sobre el **conjunto completo** de la
población analítica de 2025 (`df_2025_analitico`). Para graficar se toma una muestra de
hasta `SAMPLE_MAX` registros cuando se necesitan datos a nivel de fila; los gráficos de
conteos/medianas por grupo se construyen sobre **agregados** (no requieren muestreo).


In [ ]:
NUM_COLS_DESC = ["salario_mensual", "edad", "antiguedad", "horas_semanales"]

def resumen_descriptivo(sdf, cols):
    filas = []
    n_total = sdf.count()
    for c in cols:
        agg = sdf.select(
            F.count(c).alias("n"),
            F.mean(c).alias("media"),
            F.expr(f"percentile_approx({c}, 0.5)").alias("mediana"),
            F.stddev(c).alias("sd"),
            F.min(c).alias("min"),
            F.max(c).alias("max"),
            F.expr(f"percentile_approx({c}, 0.25)").alias("p25"),
            F.expr(f"percentile_approx({c}, 0.75)").alias("p75"),
            F.expr(f"percentile_approx({c}, 0.95)").alias("p95"),
        ).first()
        filas.append((c,) + tuple(agg))
    return pd.DataFrame(filas, columns=["variable", "n", "media", "mediana", "sd",
                                         "min", "max", "p25", "p75", "p95"])

resumen_2025 = resumen_descriptivo(df_2025_analitico, NUM_COLS_DESC)
resumen_2025


In [ ]:
# --- Muestra para graficar (hasta SAMPLE_MAX registros; las metricas ya se calcularon arriba) ---
n_analitico_2025 = df_2025_analitico.count()
frac = min(1.0, SAMPLE_MAX / n_analitico_2025)
muestra_2025_pdf = df_2025_analitico.sample(withReplacement=False, fraction=frac, seed=SEED).toPandas()
print(f"Muestra para graficar: {len(muestra_2025_pdf)} de {n_analitico_2025} registros analíticos")


### ¿Cómo se distribuyen los registros entre categorías ocupacionales, niveles educativos y dominios?

In [ ]:
# Distribuciones: se calculan como AGREGADOS sobre el conjunto completo (no la muestra)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

conteo_cat = (df_2025_analitico.groupBy("categoria_ocupacional").count()
              .orderBy("categoria_ocupacional").toPandas())
axes[0].bar(conteo_cat["categoria_ocupacional"], conteo_cat["count"])
axes[0].set_title("Registros por categoría ocupacional")
axes[0].set_xlabel("categoria_ocupacional")

conteo_edu = (df_2025_analitico.groupBy("nivel_educativo").count()
              .orderBy("nivel_educativo").toPandas())
axes[1].bar(conteo_edu["nivel_educativo"], conteo_edu["count"])
axes[1].set_title("Registros por nivel educativo")
axes[1].set_xlabel("nivel_educativo")
axes[1].tick_params(axis="x", rotation=45)

conteo_dom = (df_2025_analitico.groupBy("dominio").count()
              .orderBy("dominio").toPandas())
axes[2].bar(conteo_dom["dominio"], conteo_dom["count"])
axes[2].set_title("Registros por dominio")
axes[2].set_xlabel("dominio")

plt.tight_layout()
plt.show()


*Interpretación:* [Completar con base en el gráfico real: qué categoría ocupacional,
nivel educativo y dominio concentran más observaciones, y si hay categorías con muy pocos
registros que limiten el análisis por grupo.]

### ¿El salario presenta una distribución simétrica o asimétrica? ¿Qué diferencia existe entre su media y su mediana?

In [ ]:
media_sal = resumen_2025.loc[resumen_2025["variable"] == "salario_mensual", "media"].values[0]
mediana_sal = resumen_2025.loc[resumen_2025["variable"] == "salario_mensual", "mediana"].values[0]
print(f"Media: {media_sal:,.2f} | Mediana: {mediana_sal:,.2f} | Diferencia: {media_sal - mediana_sal:,.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(muestra_2025_pdf["salario_mensual"], bins=50)
axes[0].axvline(media_sal, color="red", linestyle="--", label="media")
axes[0].axvline(mediana_sal, color="green", linestyle="--", label="mediana")
axes[0].set_title("Salario mensual (escala original, quetzales)")
axes[0].legend()

axes[1].hist(np.log(muestra_2025_pdf["salario_mensual"]), bins=50)
axes[1].set_title("log(Salario mensual) — SOLO para visualizar\n(el modelado usa quetzales, no log)")
plt.tight_layout()
plt.show()


*Interpretación:* [Completar: si media > mediana, hay sesgo a la derecha (cola de
salarios altos); describir la magnitud de la diferencia y qué implica para elegir métricas
robustas como la mediana en comparaciones entre grupos.]

### ¿Cómo varía el salario mediano entre niveles educativos y categorías ocupacionales?

In [ ]:
mediana_por_edu = (df_2025_analitico.groupBy("nivel_educativo")
                    .agg(F.expr("percentile_approx(salario_mensual, 0.5)").alias("salario_mediano"),
                         F.count("*").alias("n"))
                    .orderBy("nivel_educativo").toPandas())

mediana_por_cat = (df_2025_analitico.groupBy("categoria_ocupacional")
                    .agg(F.expr("percentile_approx(salario_mensual, 0.5)").alias("salario_mediano"),
                         F.count("*").alias("n"))
                    .orderBy("categoria_ocupacional").toPandas())

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar(mediana_por_edu["nivel_educativo"], mediana_por_edu["salario_mediano"])
axes[0].set_title("Salario mediano por nivel educativo")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(mediana_por_cat["categoria_ocupacional"], mediana_por_cat["salario_mediano"])
axes[1].set_title("Salario mediano por categoría ocupacional")
plt.tight_layout()
plt.show()

mediana_por_edu


*Interpretación:* [Completar: describir el patrón — ¿a mayor nivel educativo, mayor
salario mediano? ¿qué categoría ocupacional tiene el salario mediano más alto/bajo, y qué
tan grande es cada grupo (columna `n`) para saber si la comparación es confiable?]

### ¿Cómo cambian el tamaño de la muestra analítica y el salario mediano entre trimestres?

In [ ]:
por_trimestre = (df_2025_analitico.groupBy("periodo_archivo")
                  .agg(F.count("*").alias("n_registros"),
                       F.expr("percentile_approx(salario_mensual, 0.5)").alias("salario_mediano"))
                  .orderBy("periodo_archivo").toPandas())

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(por_trimestre["periodo_archivo"], por_trimestre["n_registros"], alpha=0.4, label="n registros")
ax1.set_ylabel("n registros")
ax2 = ax1.twinx()
ax2.plot(por_trimestre["periodo_archivo"], por_trimestre["salario_mediano"], color="red", marker="o", label="salario mediano")
ax2.set_ylabel("salario mediano (Q)")
fig.suptitle("Tamaño de muestra y salario mediano por trimestre (2025)")
plt.tight_layout()
plt.show()

por_trimestre


*Interpretación:* [Completar con base en los valores reales: ¿el tamaño de la muestra
analítica es relativamente estable entre trimestres? ¿hay una tendencia clara al alza/baja
en el salario mediano, o fluctúa sin patrón?]

## 3. Relaciones entre variables numéricas — (5 pts)

Se calcula la correlación de Pearson entre `salario_mensual`, `edad`, `antiguedad` y
`horas_semanales` usando `VectorAssembler` + `Correlation.corr` de `pyspark.ml.stat`,
sobre **todos los registros elegibles de 2025** (sin muestreo).


In [ ]:
CORR_COLS = ["salario_mensual", "edad", "antiguedad", "horas_semanales"]

df_corr_input = df_2025_analitico.select(*CORR_COLS).na.drop()

va_corr = VectorAssembler(inputCols=CORR_COLS, outputCol="features_corr")
vdf_corr = va_corr.transform(df_corr_input)

corr_matrix = Correlation.corr(vdf_corr, "features_corr", "pearson").head()[0].toArray()
corr_pdf = pd.DataFrame(corr_matrix, index=CORR_COLS, columns=CORR_COLS)
corr_pdf


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(CORR_COLS))); ax.set_xticklabels(CORR_COLS, rotation=45, ha="right")
ax.set_yticks(range(len(CORR_COLS))); ax.set_yticklabels(CORR_COLS)
for i in range(len(CORR_COLS)):
    for j in range(len(CORR_COLS)):
        ax.text(j, i, f"{corr_matrix[i, j]:.2f}", ha="center", va="center", color="black")
ax.set_title("Matriz de correlación (Pearson) — 2025")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


### Respuestas — Ejercicio 3

**¿Qué variables presentan mayor asociación lineal con el salario?**
[Completar con los valores reales de la matriz: identificar la(s) variable(s) con el
coeficiente de correlación más alto en valor absoluto respecto a `salario_mensual`, y
comentar si la asociación es débil, moderada o fuerte según su magnitud.]

**¿Existe relación entre edad y antigüedad?**
[Completar: reportar el coeficiente `corr(edad, antiguedad)` y explicar que una correlación
positiva es esperable, ya que a mayor edad hay más oportunidad de haber acumulado años en
el puesto actual, aunque la relación no es determinística (alguien mayor puede tener un
empleo reciente).]


## 4. Segmentación de perfiles mediante KMeans — (10 pts)

**Variables de clustering:** `edad`, `antiguedad`, `horas_semanales`.

**¿Vale la pena incluir salario?** Se prueba también una variante que incluye
`salario_mensual` para decidir, como equipo, si aporta separación adicional a los perfiles
o si domina la solución por su escala/asimetría (aun estandarizando, su alta dispersión y
cola larga puede hacer que los clusters terminen definidos casi exclusivamente por nivel de
ingreso en vez de por el perfil laboral/demográfico). La decisión final y su justificación
se documentan después de comparar ambas variantes.

Todas las variables se estandarizan (media 0, desviación 1) con `StandardScaler` antes de
correr `KMeans`, para que ninguna domine por su escala.


In [ ]:
CLUSTER_COLS_BASE = ["edad", "antiguedad", "horas_semanales"]
CLUSTER_COLS_CON_SALARIO = CLUSTER_COLS_BASE + ["salario_mensual"]

def preparar_features_cluster(sdf, cols):
    d = sdf.select(*cols).na.drop()
    va = VectorAssembler(inputCols=cols, outputCol="features_raw")
    d = va.transform(d)
    scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled",
                             withMean=True, withStd=True)
    scaler_model = scaler.fit(d)
    d = scaler_model.transform(d)
    return d

def evaluar_k(d, ks=(2, 3, 4, 5)):
    resultados = []
    for k in ks:
        km = KMeans(featuresCol="features_scaled", predictionCol="cluster", k=k, seed=SEED)
        modelo = km.fit(d)
        pred = modelo.transform(d)
        evaluator = ClusteringEvaluator(featuresCol="features_scaled", predictionCol="cluster")
        silhouette = evaluator.evaluate(pred)
        resultados.append({"k": k, "silhouette": silhouette, "modelo": modelo, "pred": pred})
    return resultados

# --- Variante A: sin salario ---
data_A = preparar_features_cluster(df_2025_analitico, CLUSTER_COLS_BASE)
resultados_A = evaluar_k(data_A)

# --- Variante B: con salario ---
data_B = preparar_features_cluster(df_2025_analitico, CLUSTER_COLS_CON_SALARIO)
resultados_B = evaluar_k(data_B)

comparacion_k = pd.DataFrame({
    "k": [r["k"] for r in resultados_A],
    "silhouette_sin_salario": [r["silhouette"] for r in resultados_A],
    "silhouette_con_salario": [r["silhouette"] for r in resultados_B],
})
comparacion_k


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(comparacion_k["k"], comparacion_k["silhouette_sin_salario"], marker="o", label="sin salario")
plt.plot(comparacion_k["k"], comparacion_k["silhouette_con_salario"], marker="o", label="con salario")
plt.xlabel("k"); plt.ylabel("silhouette score")
plt.title("Silhouette por número de clusters")
plt.legend()
plt.show()


**Criterio de selección del equipo:** se elige el modelo (variante y k) con el **mayor
silhouette score**, siempre que produzca clusters de tamaño razonable (ningún cluster con
una fracción mínima de registros que lo haga poco interpretable) y perfiles claramente
distinguibles en sus medias. [Completar con la decisión real una vez corrido sobre los
datos verdaderos: qué variante y qué k se eligió, y por qué.]

In [ ]:
# --- Seleccionar el mejor modelo según el criterio anterior (ajustar índice/variante tras revisar resultados) ---
MEJOR_VARIANTE = "sin_salario"   # o "con_salario", según lo que decida el equipo
MEJOR_K = int(comparacion_k.loc[comparacion_k["silhouette_sin_salario"].idxmax(), "k"])

mejor_resultado = [r for r in resultados_A if r["k"] == MEJOR_K][0]
pred_final = mejor_resultado["pred"]
cols_perfil = CLUSTER_COLS_BASE

print(f"Modelo seleccionado: {MEJOR_VARIANTE}, k = {MEJOR_K}, silhouette = {mejor_resultado['silhouette']:.4f}")


In [ ]:
# --- Perfil de cada cluster: medias de las variables numéricas + tamaño ---
perfil_num = (pred_final.groupBy("cluster")
              .agg(F.count("*").alias("n"),
                   *[F.mean(c).alias(f"media_{c}") for c in cols_perfil])
              .orderBy("cluster").toPandas())
perfil_num


In [ ]:
# --- Composición categórica por cluster (para enriquecer la descripción de cada perfil) ---
# Se hace join por posición usando un id auxiliar, ya que features_scaled no conserva las categóricas.
# Alternativa más simple: volver a unir por índice de fila usando monotonically_increasing_id ANTES del assembler.
# Aquí se recalcula con id para poder cruzar cluster con nivel_educativo / categoria_ocupacional.

base_con_id = df_2025_analitico.select(
    F.monotonically_increasing_id().alias("_row_id"), *cols_perfil,
    "nivel_educativo", "categoria_ocupacional", "salario_mensual"
).na.drop(subset=cols_perfil)

va_id = VectorAssembler(inputCols=cols_perfil, outputCol="features_raw")
base_id_feat = va_id.transform(base_con_id)
scaler_id = StandardScaler(inputCol="features_raw", outputCol="features_scaled",
                            withMean=True, withStd=True).fit(base_id_feat)
base_id_feat = scaler_id.transform(base_id_feat)

km_final = KMeans(featuresCol="features_scaled", predictionCol="cluster", k=MEJOR_K, seed=SEED)
modelo_final = km_final.fit(base_id_feat)
pred_con_categoricas = modelo_final.transform(base_id_feat)

composicion_edu = (pred_con_categoricas.groupBy("cluster", "nivel_educativo").count()
                    .orderBy("cluster", "nivel_educativo").toPandas())
salario_por_cluster = (pred_con_categoricas.groupBy("cluster")
                        .agg(F.expr("percentile_approx(salario_mensual, 0.5)").alias("salario_mediano"),
                             F.count("*").alias("n"))
                        .orderBy("cluster").toPandas())
salario_por_cluster


### Descripción de los clusters

[Completar con base en `perfil_num`, `composicion_edu` y `salario_por_cluster` reales.
Para cada cluster, describir en una o dos frases el perfil típico, por ejemplo:

- **Cluster 0:** trabajadores jóvenes, poca antigüedad, jornada completa — salario mediano
  más bajo.
- **Cluster 1:** trabajadores de mayor edad y antigüedad, jornada completa — salario
  mediano más alto.
- **Cluster 2:** ...

Recordar: `salario_mensual` **no** se usó como predictor en los modelos supervisados ni
como parte de las features finales de clustering si el equipo decidió excluirlo; aquí solo
se usa para *describir* los clusters resultantes, lo cual es válido.]


---
## Pendiente para la entrega final (domingo 27/09) — Ejercicios 5 a 8

Las siguientes secciones se desarrollarán en la entrega final y no forman parte de este
avance:

- **5. Pipeline de regresión lineal** (20 pts): `StringIndexer` + `OneHotEncoder` para
  `nivel_educativo`, `categoria_ocupacional`, `dominio`; `VectorAssembler` con `edad`,
  `antiguedad`, `horas_semanales` + categóricas codificadas; estandarización;
  `LinearRegression`; comparación de al menos 2 configuraciones de regularización.
- **6. Pipeline de Random Forest** (20 pts): mismo esquema de features (sin estandarizar);
  `RandomForestRegressor`; comparación de al menos 2 configuraciones (árboles/profundidad).
- **7. Entrenamiento final y evaluación en 2026** (20 pts): aplicar la mejor configuración
  de cada algoritmo sobre `df_2026_analitico` y comparar métricas.
- **8. Visualización y análisis de errores** (15 pts): real vs. predicho, residuos vs.
  predicho, MAE/error medio por nivel educativo y dominio, y discusión final.
